data loading


In [1]:
import kagglehub
import pandas as pd
import os
import random
from datasets import Dataset
import datasets 
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
from transformers import TrainingArguments,Trainer



# Download and get the dataset path
path = kagglehub.dataset_download("shanegerami/ai-vs-human-text")
csv_path = os.path.join(path, "AI_Human.csv")

# Load into DataFrame
df = pd.read_csv(csv_path)







C:\Users\Sanchit\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data cleaning 


In [2]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
class Text:
    def __init__(self, text, generated):
        self.generated = generated
        self.text = text

# Create a list of Text objects from the DataFrame
data = [Text(row['text'], row['generated']) for _, row in df.iterrows()]

#So, for _, row in df.iterrows() means "for each row in the DataFrame, ignore the index, and use the row data."

ai_text = list(filter(lambda x : x.generated == 1, data))
human_text = list(filter(lambda x : x.generated == 0 , data))
human_text_shrunk = human_text[:len(ai_text)]
data = ai_text + human_text_shrunk
random.shuffle(data)
output = [t.generated for t in data]
texts = [t.text for t in data]
new_df = pd.DataFrame({'text': texts, 'label': output})


# convert to a hugging face dataset

dataset = Dataset.from_pandas(new_df)


initalize the tockenizer and the model

In [3]:
# Use a pipeline as a high-level helper
# from transformers import pipeline

# pipe = pipeline("fill-mask", model="distilbert/distilbert-base-uncased")(faster method but for testing model only not for fine tuning it )

# Load model directly


tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased",num_labels = 2,problem_type = "single_label_classification")




Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


split data and tokenize data



In [4]:
dataset = dataset.train_test_split(test_size=0.2)
train_ds = dataset['train']
test_ds = dataset['test']


def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding=True,return_tensors='pt')



tokenized_train = train_ds.map(tokenize, batched=True)
tokenized_test = test_ds.map(tokenize, batched=True)

tokenized_train.set_format('torch')
tokenized_test.set_format('torch')





# can also initlaize directly using lambda

# batch_size = 50000
# all_tokenized = []
# for i in range(0, len(data), batch_size):
#     texts = [item.text for item in data[i:i+batch_size]]
#     tokenized = tokenizer(
#         texts,
#         padding=True,
#         truncation=True,
#         return_tensors="pt"
#     )
#     all_tokenized.append(tokenized)



Map: 100%|██████████| 72576/72576 [00:24<00:00, 2922.47 examples/s]


changes for insert into the model 

In [5]:
tokenized_train = tokenized_train.rename_column('label','labels')
tokenized_train = tokenized_train.remove_columns('text')
tokenized_test = tokenized_test.rename_column('label','labels')
tokenized_test = tokenized_test.remove_columns('text')
print(tokenized_train[8])


{'labels': tensor(0.), 'input_ids': tensor([  101,  8448,  4054,  1010,  2493,  2323,  4685,  2451,  2326,  1046,
         2015,  2005,  2009,  3632,  1046,  2146,  2126,  1012,  5094,  2115,
         2451,  3084,  1046, 14910,  5063,  3560,  4489,  1999,  2061,  2116,
         3572,  1012,  1049, 22895,  2100,  2111,  2031,  2589,  2451,  2326,
         1998,  2027,  2145,  2134,  1005,  1056,  3335,  2037,  5440,  2547,
         2265,  1010,  4164,  1010,  2030,  1046,  2283,  1012,  2216,  2477,
         2097,  2272,  1998,  2175,  2087,  5791,  2021,  2054,  2204,  2017,
         2079,  1999,  2115,  2166,  2085,  1039, 22895,  2202,  2017,  1046,
         2146,  2126,  1012,  1045,  2031,  2589,  2451,  2326,  1046,  3232,
         1997,  2335,  1998,  2009,  2001,  2307,  1012,  1045,  2253,  2105,
         1998,  4711,  2185, 19070, 11669,  1999,  2379,  2011, 11681,  1998,
         3067,  1012,  2125,  2607,  1010,  2009,  2001,  2051,  1011, 15077,
         1010,  2021,  2009,

set up evaluation metrics


In [6]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }
    


set up trainer


In [7]:
# Training Arguments
args = TrainingArguments(
    output_dir="model_output",
    eval_strategy="epoch",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=2e-5,
    
)

#  Create Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)


C:\Users\Sanchit\AppData\Local\Temp\ipykernel_4976\1318021941.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

C:\Users\Sanchit\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
